# Fase 5 · M01c: Modelos Lineales — Extendido

**TFM: Pronóstico del Éxito y del Abandono en los Títulos de Grado de la UJI**

| | |
|---|---|
| **Autora** | María José Morte Ruiz |
| **Institución** | UOC + Universitat Jaume I |
| **Email** | mjmorteruiz@uoc.edu · morte@uji.es |
| **Fase** | 5 — Modelado Clásico |
| **Módulo** | M01c — Modelos Lineales (Extendido) |

---

## 🎯 Qué hace

Entrena y evalúa la familia extendida de modelos lineales sobre el dataset D_strict
(24 features + target `abandono`, incl. indicadores de imputación informativa).
Para cada modelo se prueban las 3 estrategias de balance de clases y se valida
con 5-Fold CV estratificado.

**Modelos de esta familia:**

| Modelo | Tipo | predict_proba | Valor diferencial |
|---|---|---|---|
| `Perceptron` | Online learning | Calibrada | Origen histórico 1958 — baseline del aprendizaje lineal |
| `SGD` | Meta-algoritmo | Nativa | Escalable, flexible, `loss=modified_huber` |
| `SGD-ElasticNet` | Regularización L1+L2 | Nativa | Generaliza Ridge (L2) añadiendo selección de features |

**Sobre RidgeClassifier — excluido con justificación:**
> *`RidgeClassifier` se omite porque su regularización L2 ya está representada
> por `LogisticRegression(penalty='l2')` en M01b, que implementa el mismo concepto
> matemático con diseño nativo para clasificación probabilística.
> `SGD-ElasticNet` supera a Ridge añadiendo la componente L1 (selección de features)
> sin necesidad de calibración post-hoc. Incluir `RidgeClassifier` sería redundancia
> metodológica que no aportaría información nueva al tribunal.*

## 📋 Requisitos

- `data/05_modelado/X_train_prep.parquet` — generado por `f5_m01a_preparacion.ipynb`
- `data/05_modelado/y_train.parquet`
- `src/` con `config.py`, `utils/`, `html/`
- Entorno: `tfm_abandono` (scikit-learn ≥ 1.3, imbalanced-learn, joblib, optuna)

## 📤 Genera

| Archivo | Contenido |
|---|---|
| `data/05_modelado/results/results_lineales_ext.parquet` | Resultados CV + test |
| `data/05_modelado/models/Perceptron__*.pkl` | 3 pipelines entrenados |
| `data/05_modelado/models/SGD__*.pkl` | 3 pipelines entrenados |
| `data/05_modelado/models/SGDElasticNet__*.pkl` | 3 pipelines entrenados |
| `docs/html/fase5/m01c_lineales_ext.html` | Informe HTML |

## 🔄 Flujo

```
X_train_prep / y_train  (f5_m01a_preparacion)
    ↓ Optuna 20 trials × 3 modelos → hiperparámetros óptimos
    ↓ Entrenamiento 9 combinaciones (3 modelos × 3 estrategias)
    ↓ CV 5-Fold estratificado → AUC, F1, Precision, Recall
    ↓ Serialización pkl + parquet resultados
    → results_lineales_ext.parquet + 9 modelos .pkl
```

## ➡️ Siguiente

`f5_m01d_lineales.ipynb` — Comparativa completa 7 modelos lineales

In [1]:
# ============================================================================
# CELDA 1: CONFIGURACIÓN Y DEPENDENCIAS
# ============================================================================
# ROOT detection robusto. Importa utilidades del proyecto.
# ============================================================================

import sys, warnings, json, time, tracemalloc
from pathlib import Path
from datetime import datetime

import joblib
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from sklearn.linear_model import Perceptron, SGDClassifier
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.pipeline import Pipeline
from sklearn.model_selection import (
    StratifiedKFold, StratifiedShuffleSplit,
    cross_validate, learning_curve
)
from sklearn.metrics import (
    f1_score, roc_auc_score, precision_score, recall_score,
    accuracy_score, cohen_kappa_score, log_loss,
    confusion_matrix, classification_report, roc_curve
)
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

try:
    from codecarbon import EmissionsTracker
    CODECARBON_OK = True
except ImportError:
    CODECARBON_OK = False
    print('⚠️  codecarbon no instalado')

try:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    OPTUNA_OK = True
except ImportError:
    OPTUNA_OK = False
    print('⚠️  optuna no instalado')

warnings.filterwarnings('ignore')

# ROOT detection
ROOT = Path.cwd()
for _ in range(6):
    if (ROOT / 'src').exists():
        break
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT))

from src.config import RUTA_HTML, info_entorno
from src.utils import crear_directorios, formato_numero_es
from src.utils.graficos import figura_a_base64
from src.html.render import render_pagina_desde_fichero

# Rutas
RUTA_MODELADO = ROOT / 'data' / '05_modelado'
RUTA_RESULTS  = RUTA_MODELADO / 'results'
RUTA_MODELS   = RUTA_MODELADO / 'models'
RUTA_HTML_F5  = RUTA_HTML / 'fase5'
crear_directorios([RUTA_RESULTS, RUTA_MODELS, RUTA_HTML_F5])

# Constantes
RANDOM_STATE    = 42
N_SPLITS_CV     = 5
N_TRIALS_OPTUNA = 20
FAMILIA         = 'lineales_ext'
ESTRATEGIAS     = ['none', 'balanced', 'smote']
MODELOS_EXT     = ['Perceptron', 'SGD', 'SGDElasticNet']
COLOR           = '#3182ce'
fmt             = formato_numero_es

graficos_b64 = {}

info_entorno()
print(f'\n📂 ROOT    : {ROOT}')
print(f'📂 RESULTS : {RUTA_RESULTS}')
print(f'📂 MODELS  : {RUTA_MODELS}')
print(f'\n🌿 codecarbon: {CODECARBON_OK}')
print(f'🔍 optuna     : {OPTUNA_OK}')

✓ Directorios verificados: 3
✓ ===========================================================================
✓ 📌 INFORMACIÓN DEL ENTORNO DEL PROYECTO
✓ ===========================================================================
✓ 🖥️  Entorno detectado: Local
✓ 📂 Ruta base:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI
✓ 📁 RAW:           C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\00_raw
✓ 📁 INTERIM:       C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\01_interim
✓ 📁 PROCESSED:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\02_processed
✓ 📁 FEATURES:      C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\03_features
✓ 📁 AUTOML:        C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\automl
✓ 📁 NOTEBOOKS:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\notebooks
✓ 📄 Excel principal: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\00_raw\datos_proyecto_sin_preinscrip.xlsx
✓

In [2]:
# ============================================================================
# CELDA 2: CARGA DE DATOS
# ============================================================================
# Lee splits preprocesados generados por f5_m01a_preparacion.
# ============================================================================

X_train = pd.read_parquet(RUTA_MODELADO / 'X_train.parquet')
X_test  = pd.read_parquet(RUTA_MODELADO / 'X_test.parquet')
y_train = pd.read_parquet(RUTA_MODELADO / 'y_train.parquet').squeeze()
y_test  = pd.read_parquet(RUTA_MODELADO / 'y_test.parquet').squeeze()

ruta_train_prep = RUTA_MODELADO / 'X_train_prep.parquet'
ruta_test_prep  = RUTA_MODELADO / 'X_test_prep.parquet'

if ruta_train_prep.exists() and ruta_test_prep.exists():
    X_train_prep = pd.read_parquet(ruta_train_prep)
    X_test_prep  = pd.read_parquet(ruta_test_prep)
    print('✅ Preprocesados cargados desde disco')
else:
    pipeline_prep = joblib.load(RUTA_MODELADO / 'pipeline_preprocesamiento.pkl')
    X_train_prep = pd.DataFrame(
        pipeline_prep.transform(X_train),
        columns=X_train.columns, index=X_train.index
    )
    X_test_prep = pd.DataFrame(
        pipeline_prep.transform(X_test),
        columns=X_test.columns, index=X_test.index
    )
    print('✅ Preprocesados generados')

FEATURE_NAMES = list(X_train_prep.columns)
N_FEATURES    = len(FEATURE_NAMES)

print(f'X_train : {fmt(len(X_train))} × {N_FEATURES}  |  abandono: {y_train.mean()*100:.1f}%')
print(f'X_test  : {fmt(len(X_test))}  × {N_FEATURES}  |  abandono: {y_test.mean()*100:.1f}%')
print(f'\n⚠️  X_test INTOCABLE hasta f5_m06_ensambles.ipynb')

✅ Preprocesados cargados desde disco
X_train : 26.896 × 27  |  abandono: 29.2%
X_test  : 6.725  × 27  |  abandono: 29.2%

⚠️  X_test INTOCABLE hasta f5_m06_ensambles.ipynb


In [3]:
# ============================================================================
# CELDA 3: FUNCIONES AUXILIARES
# ============================================================================

CV = StratifiedKFold(n_splits=N_SPLITS_CV, shuffle=True, random_state=RANDOM_STATE)


def construir_pipeline(modelo, estrategia: str):
    if estrategia == 'smote':
        return ImbPipeline([
            ('smote', SMOTE(random_state=RANDOM_STATE)),
            ('model', modelo)
        ])
    return Pipeline([('model', modelo)])


def evaluar_cv(nombre: str, modelo, X, y, estrategia: str) -> dict:
    pipe = construir_pipeline(modelo, estrategia)
    scoring = {'f1': 'f1', 'roc_auc': 'roc_auc',
               'precision': 'precision', 'recall': 'recall', 'accuracy': 'accuracy'}
    tracemalloc.start()
    t0     = time.time()
    cv_res = cross_validate(pipe, X, y, cv=CV, scoring=scoring,
                            return_train_score=False, n_jobs=-1)
    t_total = time.time() - t0
    _, mem_pico = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    return {
        'modelo'         : nombre,
        'estrategia'     : estrategia,
        'familia'        : FAMILIA,
        'f1_mean'        : cv_res['test_f1'].mean(),
        'f1_std'         : cv_res['test_f1'].std(),
        'auc_mean'       : cv_res['test_roc_auc'].mean(),
        'auc_std'        : cv_res['test_roc_auc'].std(),
        'precision_mean' : cv_res['test_precision'].mean(),
        'recall_mean'    : cv_res['test_recall'].mean(),
        'accuracy_mean'  : cv_res['test_accuracy'].mean(),
        'tiempo_s'       : round(t_total, 3),
        'memoria_mb'     : round(mem_pico / 1024**2, 2),
    }


def calcular_metricas_test(nombre: str, pipe_fit, X_te, y_te, estrategia: str) -> dict:
    y_pred  = pipe_fit.predict(X_te)
    y_proba = (
        pipe_fit.predict_proba(X_te)[:, 1]
        if hasattr(pipe_fit, 'predict_proba')
        else pipe_fit.decision_function(X_te)
    )
    if y_proba.min() < 0 or y_proba.max() > 1:
        y_proba = (y_proba - y_proba.min()) / (y_proba.max() - y_proba.min() + 1e-9)
    return {
        'modelo'         : nombre,
        'estrategia'     : estrategia,
        'f1_test'        : round(f1_score(y_te, y_pred), 4),
        'auc_test'       : round(roc_auc_score(y_te, y_proba), 4),
        'precision_test' : round(precision_score(y_te, y_pred), 4),
        'recall_test'    : round(recall_score(y_te, y_pred), 4),
        'accuracy_test'  : round(accuracy_score(y_te, y_pred), 4),
        'kappa_test'     : round(cohen_kappa_score(y_te, y_pred), 4),
        'logloss_test'   : round(log_loss(y_te, y_proba), 4),
    }


def construir_modelo(nombre: str, estrategia: str):
    p  = PARAMS_OPTUNA[nombre]
    cw = 'balanced' if estrategia == 'balanced' else None

    if nombre == 'Perceptron':
        base = Perceptron(
            penalty=p.get('penalty', 'l2'),
            alpha=p.get('alpha', 0.001),
            max_iter=1000,
            class_weight=cw,
            random_state=RANDOM_STATE
        )
        # Perceptron no tiene predict_proba — calibración con Platt scaling
        return CalibratedClassifierCV(base, cv=3, method='sigmoid')

    elif nombre == 'SGD':
        return SGDClassifier(
            loss='modified_huber',   # da predict_proba nativa
            penalty=p.get('penalty', 'l2'),
            alpha=p.get('alpha', 0.0001),
            learning_rate=p.get('learning_rate', 'optimal'),
            max_iter=1000,
            class_weight=cw,
            random_state=RANDOM_STATE
        )

    elif nombre == 'SGDElasticNet':
        # SGD con penalización ElasticNet (L1+L2) — generaliza Ridge (L2 puro)
        # loss='log_loss' da predict_proba nativa sin calibración
        return SGDClassifier(
            loss='log_loss',
            penalty='elasticnet',
            l1_ratio=p.get('l1_ratio', 0.5),  # 0=L2 puro, 1=L1 puro, 0.5=balance
            alpha=p.get('alpha', 0.0001),
            learning_rate=p.get('learning_rate', 'optimal'),
            max_iter=1000,
            class_weight=cw,
            random_state=RANDOM_STATE
        )


print('✅ Funciones auxiliares definidas')
print('   · construir_pipeline  · evaluar_cv')
print('   · calcular_metricas_test  · construir_modelo')

✅ Funciones auxiliares definidas
   · construir_pipeline  · evaluar_cv
   · calcular_metricas_test  · construir_modelo


In [4]:
# ============================================================================
# CELDA 4: HIPERPARÁMETROS ÓPTIMOS (OPTUNA)
# ============================================================================
# Para re-ejecutar la búsqueda: cambiar FORZAR_OPTUNA = True.
# Perceptron y SGD: hiperparámetros trasladados de versión anterior.
# SGDElasticNet: nuevos — búsqueda Optuna 20 trials.
# ============================================================================

FORZAR_OPTUNA = False

PARAMS_OPTUNA = {
    'Perceptron'   : {'penalty': 'l2',   'alpha': 0.001},
    'SGD'          : {'penalty': 'l2',   'alpha': 0.0001, 'learning_rate': 'optimal',
                      'loss': 'modified_huber'},
    'SGDElasticNet': {'l1_ratio': 0.5,   'alpha': 0.0001, 'learning_rate': 'optimal'},
}

AUC_OPTUNA = {
    'Perceptron'   : None,  # se actualiza tras primera ejecución
    'SGD'          : None,
    'SGDElasticNet': None,
}

if FORZAR_OPTUNA and OPTUNA_OK:
    print('🔍 Ejecutando búsqueda Optuna...')

    def objetivo_perceptron(trial):
        penalty = trial.suggest_categorical('penalty', ['l2', 'l1', 'elasticnet'])
        alpha   = trial.suggest_float('alpha', 1e-4, 1e-1, log=True)
        modelo  = CalibratedClassifierCV(
            Perceptron(penalty=penalty, alpha=alpha, max_iter=1000,
                       random_state=RANDOM_STATE), cv=3)
        pipe    = construir_pipeline(modelo, 'balanced')
        cv_res  = cross_validate(pipe, X_train_prep, y_train, cv=CV,
                                 scoring='roc_auc', n_jobs=-1)
        return cv_res['test_score'].mean()

    def objetivo_sgd(trial):
        alpha = trial.suggest_float('alpha', 1e-5, 1e-2, log=True)
        lr    = trial.suggest_categorical('learning_rate', ['optimal', 'adaptive'])
        modelo = SGDClassifier(loss='modified_huber', penalty='l2',
                               alpha=alpha, learning_rate=lr,
                               max_iter=1000, random_state=RANDOM_STATE)
        pipe   = construir_pipeline(modelo, 'balanced')
        cv_res = cross_validate(pipe, X_train_prep, y_train, cv=CV,
                                scoring='roc_auc', n_jobs=-1)
        return cv_res['test_score'].mean()

    def objetivo_sgd_en(trial):
        l1_ratio = trial.suggest_float('l1_ratio', 0.1, 0.9)
        alpha    = trial.suggest_float('alpha', 1e-5, 1e-2, log=True)
        lr       = trial.suggest_categorical('learning_rate', ['optimal', 'adaptive'])
        modelo   = SGDClassifier(loss='log_loss', penalty='elasticnet',
                                 l1_ratio=l1_ratio, alpha=alpha,
                                 learning_rate=lr, max_iter=1000,
                                 random_state=RANDOM_STATE)
        pipe   = construir_pipeline(modelo, 'balanced')
        cv_res = cross_validate(pipe, X_train_prep, y_train, cv=CV,
                                scoring='roc_auc', n_jobs=-1)
        return cv_res['test_score'].mean()

    for nombre, fn in [('Perceptron', objetivo_perceptron),
                       ('SGD',        objetivo_sgd),
                       ('SGDElasticNet', objetivo_sgd_en)]:
        study = optuna.create_study(direction='maximize',
                                    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
        study.optimize(fn, n_trials=N_TRIALS_OPTUNA, show_progress_bar=False)
        PARAMS_OPTUNA[nombre].update(study.best_params)
        AUC_OPTUNA[nombre] = round(study.best_value, 4)
        print(f'  ✅ {nombre}: AUC={AUC_OPTUNA[nombre]:.4f} → {study.best_params}')

print('✅ Hiperparámetros configurados')
for nombre, params in PARAMS_OPTUNA.items():
    auc_str = f'AUC={AUC_OPTUNA[nombre]:.4f}' if AUC_OPTUNA[nombre] else 'AUC=pendiente'
    print(f'   {nombre:<16}: {auc_str}  →  {params}')

✅ Hiperparámetros configurados
   Perceptron      : AUC=pendiente  →  {'penalty': 'l2', 'alpha': 0.001}
   SGD             : AUC=pendiente  →  {'penalty': 'l2', 'alpha': 0.0001, 'learning_rate': 'optimal', 'loss': 'modified_huber'}
   SGDElasticNet   : AUC=pendiente  →  {'l1_ratio': 0.5, 'alpha': 0.0001, 'learning_rate': 'optimal'}


In [5]:
# ============================================================================
# CELDA 4b: LIMPIEZA DE PKL OBSOLETOS
# ============================================================================
# Elimina pkl de esta familia si fueron entrenados con distinto nº de features.
# Lógica robusta: solo borra si PUEDE verificar y el nº es incorrecto.
# ============================================================================

import glob, os, json as _json, joblib as _jl

ruta_meta = RUTA_MODELADO / 'meta_preparacion.json'
n_features_actual = None
if ruta_meta.exists():
    with open(ruta_meta) as _f:
        n_features_actual = _json.load(_f).get('n_features_final')

eliminados = []
for pkl in glob.glob(str(RUTA_MODELS / '*.pkl')):
    nombre = os.path.basename(pkl)
    if any(m in nombre for m in MODELOS_EXT):
        try:
            modelo = _jl.load(pkl)
            ultimo = modelo.steps[-1][1] if hasattr(modelo, 'steps') else modelo
            n_pkl = getattr(ultimo, 'n_features_in_', None)
            # Buscar en estructura calibrada si es necesario
            if n_pkl is None and hasattr(ultimo, 'estimator'):
                n_pkl = getattr(ultimo.estimator, 'n_features_in_', None)
            if n_pkl is None and hasattr(ultimo, 'calibrated_classifiers_'):
                try:
                    n_pkl = getattr(ultimo.calibrated_classifiers_[0].estimator,
                                    'n_features_in_', None)
                except Exception:
                    pass
            if n_pkl is not None and n_features_actual is not None and n_pkl != n_features_actual:
                os.remove(pkl)
                eliminados.append(f'{nombre} ({n_pkl}→obsoleto)')
            elif n_pkl is None:
                print(f'  ⚠️  {nombre}: no verificable — conservado')
        except Exception:
            print(f'  ⚠️  {nombre}: no cargable — conservado (no se borra)')

# Si faltan pkl → borrar parquet para forzar re-entrenamiento completo
claves_esperadas = [f'{m}__{e}' for m in MODELOS_EXT for e in ESTRATEGIAS]
pkls_presentes   = {os.path.basename(p).replace('.pkl','')
                    for p in glob.glob(str(RUTA_MODELS / '*.pkl'))}
faltan = [c for c in claves_esperadas if c not in pkls_presentes]

if faltan:
    print(f'⚠️  PKL faltantes: {faltan}')
    ruta_parquet = RUTA_RESULTS / 'results_lineales_ext.parquet'
    if ruta_parquet.exists():
        os.remove(ruta_parquet)
        print('   Parquet eliminado → re-entrenamiento completo')

if eliminados:
    print(f'⚠️  PKL obsoletos eliminados: {eliminados}')
else:
    print(f'✅ PKL OK con {n_features_actual} features')

⚠️  PKL faltantes: ['Perceptron__none', 'Perceptron__balanced', 'Perceptron__smote', 'SGD__none', 'SGD__balanced', 'SGD__smote', 'SGDElasticNet__none', 'SGDElasticNet__balanced', 'SGDElasticNet__smote']
   Parquet eliminado → re-entrenamiento completo
✅ PKL OK con 27 features


In [6]:
# ============================================================================
# CELDA 5: ENTRENAMIENTO — 9 COMBINACIONES (3 modelos × 3 estrategias)
# ============================================================================
# Si los pkl existen en disco: los carga directamente.
# Si faltan: entrena solo los que faltan.
# ============================================================================

claves_esperadas = [f'{m}__{e}' for m in MODELOS_EXT for e in ESTRATEGIAS]
pkls_en_disco    = [c for c in claves_esperadas
                    if (RUTA_MODELS / f'{c}.pkl').exists()]
parquet_en_disco = (RUTA_RESULTS / 'results_lineales_ext.parquet').exists()

print(f'📦 Modelos en disco : {len(pkls_en_disco)}/9')
print(f'📊 Parquet resultados: {parquet_en_disco}')

modelos_fit = {c: joblib.load(RUTA_MODELS / f'{c}.pkl') for c in pkls_en_disco}

claves_pendientes = [c for c in claves_esperadas if c not in modelos_fit]

if claves_pendientes:
    print(f'\n⏳ Entrenando {len(claves_pendientes)} combinaciones pendientes...')
    if CODECARBON_OK:
        tracker = EmissionsTracker(project_name='TFM_f5_m01c_ext',
                                   output_dir=str(RUTA_RESULTS), log_level='error')
        tracker.start()
    for clave in claves_pendientes:
        nombre, estrategia = clave.split('__')
        pipe = construir_pipeline(construir_modelo(nombre, estrategia), estrategia)
        pipe.fit(X_train_prep, y_train)
        modelos_fit[clave] = pipe
        joblib.dump(pipe, RUTA_MODELS / f'{clave}.pkl')
        print(f'   ✅ {clave}')
    if CODECARBON_OK:
        emisiones = tracker.stop()
        print(f'🌿 CO₂: {emisiones:.6f} kg')
    else:
        emisiones = None
else:
    emisiones = None
    print('✅ Todos los modelos ya estaban en disco')

# Construir df_cv
if parquet_en_disco:
    df_res = pd.read_parquet(RUTA_RESULTS / 'results_lineales_ext.parquet')
    df_cv  = df_res.sort_values('auc_mean', ascending=False)
    print('✅ Resultados cargados desde parquet')
else:
    print('\n⏳ Calculando métricas CV...')
    resultados_cv, resultados_test = [], []
    for nombre in MODELOS_EXT:
        print(f'   {nombre}...', end=' ', flush=True)
        for estrategia in ESTRATEGIAS:
            res_cv   = evaluar_cv(nombre, construir_modelo(nombre, estrategia),
                                  X_train_prep, y_train, estrategia)
            res_test = calcular_metricas_test(
                nombre, modelos_fit[f'{nombre}__{estrategia}'],
                X_test_prep, y_test, estrategia)
            resultados_cv.append(res_cv)
            resultados_test.append(res_test)
        print('✅')
    df_cv   = pd.DataFrame(resultados_cv).sort_values('auc_mean', ascending=False)
    df_test = pd.DataFrame(resultados_test)
    df_res  = df_cv.merge(df_test, on=['modelo', 'estrategia'])

MEJOR_CLAVE      = f'{df_cv.iloc[0]["modelo"]}__{df_cv.iloc[0]["estrategia"]}'
MEJOR_MODELO     = df_cv.iloc[0]['modelo']
MEJOR_ESTRATEGIA = df_cv.iloc[0]['estrategia']
mejor_pipe       = modelos_fit[MEJOR_CLAVE]

print(f'\n🏆 MEJOR EXT: {MEJOR_MODELO} + {MEJOR_ESTRATEGIA}')
print(f'   AUC CV = {df_cv.iloc[0]["auc_mean"]:.4f}  |  F1 CV = {df_cv.iloc[0]["f1_mean"]:.4f}')

[codecarbon WARNING @ 17:51:25] Multiple instances of codecarbon are allowed to run at the same time.


📦 Modelos en disco : 0/9
📊 Parquet resultados: False

⏳ Entrenando 9 combinaciones pendientes...
   ✅ Perceptron__none
   ✅ Perceptron__balanced
   ✅ Perceptron__smote
   ✅ SGD__none
   ✅ SGD__balanced
   ✅ SGD__smote
   ✅ SGDElasticNet__none
   ✅ SGDElasticNet__balanced
   ✅ SGDElasticNet__smote
🌿 CO₂: 0.000009 kg

⏳ Calculando métricas CV...
✅  Perceptron... 
✅  SGD... 
✅  SGDElasticNet... 

🏆 MEJOR EXT: SGDElasticNet + smote
   AUC CV = 0.9080  |  F1 CV = 0.7499


In [7]:
# ============================================================================
# CELDA 6: TABLA DE RESULTADOS
# ============================================================================

cols_cv = ['modelo', 'estrategia', 'auc_mean', 'auc_std',
           'f1_mean', 'precision_mean', 'recall_mean', 'tiempo_s']

print('=' * 70)
print('RESULTADOS 5-Fold CV — ordenado por AUC')
print('=' * 70)
print(df_cv[cols_cv].to_string(index=False, float_format='{:.4f}'.format))

if 'auc_test' in df_res.columns:
    cols_test = ['modelo', 'estrategia', 'auc_test', 'f1_test',
                 'precision_test', 'recall_test', 'kappa_test']
    print('\n' + '=' * 70)
    print('MÉTRICAS EN TEST (informativas — selección por CV)')
    print('=' * 70)
    print(df_res[cols_test].sort_values('auc_test', ascending=False)
          .to_string(index=False, float_format='{:.4f}'.format))

RESULTADOS 5-Fold CV — ordenado por AUC
       modelo estrategia  auc_mean  auc_std  f1_mean  precision_mean  recall_mean  tiempo_s
SGDElasticNet      smote    0.9080   0.0035   0.7499          0.6817       0.8333    4.6230
SGDElasticNet   balanced    0.9059   0.0049   0.7473          0.6853       0.8232    0.7640
SGDElasticNet       none    0.9052   0.0039   0.7394          0.7825       0.7018    0.8220
          SGD      smote    0.9039   0.0058   0.7359          0.6492       0.8499    3.0930
          SGD       none    0.8997   0.0042   0.7267          0.7865       0.6805   18.5320
          SGD   balanced    0.8904   0.0127   0.7262          0.6575       0.8190    2.5400
   Perceptron      smote    0.8832   0.0071   0.7159          0.6531       0.7924    8.7650
   Perceptron   balanced    0.8694   0.0219   0.6320          0.7955       0.5261    8.1060
   Perceptron       none    0.8693   0.0092   0.6225          0.8075       0.5100    5.0420

MÉTRICAS EN TEST (informativas — selecc

In [8]:
# ============================================================================
# CELDA 7: ANÁLISIS DE COEFICIENTES
# ============================================================================
# SGD y SGDElasticNet tienen coeficientes interpretables.
# SGDElasticNet con l1_ratio > 0 puede llevar algunos coeficientes a 0
# (selección automática de features) — diferencial respecto a Ridge/L2 puro.
# ============================================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, nombre_m in zip(axes, ['SGD', 'SGDElasticNet']):
    mejor_est = (df_cv[df_cv['modelo'] == nombre_m]
                 .sort_values('auc_mean', ascending=False).iloc[0]['estrategia'])
    pipe = modelos_fit[f'{nombre_m}__{mejor_est}']
    
    # Extraer modelo base
    modelo_base = pipe.steps[-1][1] if hasattr(pipe, 'steps') else pipe
    
    if hasattr(modelo_base, 'coef_'):
        coef = pd.Series(modelo_base.coef_.ravel(), index=FEATURE_NAMES)
        n_cero = (coef.abs() < 1e-6).sum()
        top10 = coef.abs().sort_values(ascending=False).head(10)
        
        colores = ['#e53e3e' if coef[f] > 0 else '#3182ce' for f in top10.index]
        ax.barh(top10.index[::-1], coef[top10.index[::-1]], color=colores[::-1])
        ax.axvline(0, color='black', linewidth=0.8)
        ax.set_title(f'{nombre_m} ({mejor_est})\n'
                     f'Features con coef≈0: {n_cero}/{len(FEATURE_NAMES)}',
                     fontsize=10, fontweight='bold')
        ax.set_xlabel('Coeficiente')
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
    else:
        ax.text(0.5, 0.5, f'{nombre_m}\nsin coeficientes\ndirectos',
                ha='center', va='center', transform=ax.transAxes)
        ax.set_title(nombre_m)

fig.suptitle('Coeficientes SGD vs SGDElasticNet — efecto de la regularización L1',
             fontsize=11, y=1.02)
plt.tight_layout()
graficos_b64['coeficientes'] = figura_a_base64(fig)
plt.close(fig)
print('✅ Gráfico coeficientes generado')
print('   Nota: coeficientes≈0 en SGDElasticNet indican features irrelevantes (selección L1)')

✅ Gráfico coeficientes generado
   Nota: coeficientes≈0 en SGDElasticNet indican features irrelevantes (selección L1)


In [9]:
# ============================================================================
# CELDA 8: CURVAS ROC + CALIBRACIÓN
# ============================================================================

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
colores_3 = [COLOR, '#e53e3e', '#38a169']

ax_roc = axes[0]
ax_roc.plot([0,1], [0,1], 'k--', linewidth=1)

ax_cal = axes[1]
ax_cal.plot([0,1], [0,1], 'k--', linewidth=1.2, label='Calibración perfecta')

for idx, nombre_m in enumerate(MODELOS_EXT):
    mejor_est = (df_cv[df_cv['modelo'] == nombre_m]
                 .sort_values('auc_mean', ascending=False).iloc[0]['estrategia'])
    pipe = modelos_fit[f'{nombre_m}__{mejor_est}']
    
    if hasattr(pipe, 'predict_proba'):
        y_proba = pipe.predict_proba(X_test_prep)[:, 1]
    else:
        scores = pipe.decision_function(X_test_prep)
        y_proba = (scores - scores.min()) / (scores.max() - scores.min() + 1e-9)
    
    auc_val = roc_auc_score(y_test, y_proba)
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    ax_roc.plot(fpr, tpr, color=colores_3[idx], linewidth=2,
                label=f'{nombre_m} (AUC={auc_val:.3f})')
    
    fraction_pos, mean_pred = calibration_curve(y_test, y_proba, n_bins=10)
    ax_cal.plot(mean_pred, fraction_pos, color=colores_3[idx],
                marker='o', linewidth=2, label=nombre_m)

ax_roc.set_xlabel('FPR'); ax_roc.set_ylabel('TPR')
ax_roc.set_title('Curvas ROC — mejor estrategia por modelo', fontweight='bold')
ax_roc.legend(fontsize=9)
ax_roc.spines['top'].set_visible(False); ax_roc.spines['right'].set_visible(False)

ax_cal.set_xlabel('Probabilidad predicha'); ax_cal.set_ylabel('Fracción positivos reales')
ax_cal.set_title('Calibración de probabilidades', fontweight='bold')
ax_cal.legend(fontsize=9)
ax_cal.spines['top'].set_visible(False); ax_cal.spines['right'].set_visible(False)

plt.tight_layout()
graficos_b64['roc_calibracion'] = figura_a_base64(fig)
plt.close(fig)
print('✅ Gráficos ROC + calibración generados')

✅ Gráficos ROC + calibración generados


In [10]:
# ============================================================================
# CELDA 9: CURVA DE APRENDIZAJE + MATRIZ DE CONFUSIÓN
# ============================================================================

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Curva aprendizaje del mejor modelo
sss = StratifiedShuffleSplit(n_splits=1, test_size=0.70, random_state=RANDOM_STATE)
for train_idx, _ in sss.split(X_train_prep, y_train):
    X_sub = X_train_prep.iloc[train_idx]
    y_sub = y_train.iloc[train_idx]

train_sizes, train_scores, val_scores = learning_curve(
    construir_pipeline(construir_modelo(MEJOR_MODELO, MEJOR_ESTRATEGIA), MEJOR_ESTRATEGIA),
    X_sub, y_sub, cv=CV, scoring='roc_auc',
    train_sizes=np.linspace(0.1, 1.0, 8), n_jobs=-1
)

ax_lc = axes[0]
ax_lc.plot(train_sizes, train_scores.mean(axis=1), color=COLOR, label='Train AUC')
ax_lc.fill_between(train_sizes,
                   train_scores.mean(axis=1) - train_scores.std(axis=1),
                   train_scores.mean(axis=1) + train_scores.std(axis=1),
                   alpha=0.15, color=COLOR)
ax_lc.plot(train_sizes, val_scores.mean(axis=1), color='#e53e3e', label='Val AUC')
ax_lc.fill_between(train_sizes,
                   val_scores.mean(axis=1) - val_scores.std(axis=1),
                   val_scores.mean(axis=1) + val_scores.std(axis=1),
                   alpha=0.15, color='#e53e3e')
ax_lc.set_xlabel('Tamaño train'); ax_lc.set_ylabel('AUC-ROC')
ax_lc.set_title(f'Curva de aprendizaje\n{MEJOR_MODELO} + {MEJOR_ESTRATEGIA}',
                fontweight='bold')
ax_lc.legend(); ax_lc.spines['top'].set_visible(False); ax_lc.spines['right'].set_visible(False)

# Matriz de confusión
y_pred_mejor = mejor_pipe.predict(X_test_prep)
cm = confusion_matrix(y_test, y_pred_mejor)
ax_cm = axes[1]
im = ax_cm.imshow(cm, interpolation='nearest', cmap='Blues')
plt.colorbar(im, ax=ax_cm)
clases = ['Continúa (0)', 'Abandona (1)']
ax_cm.set_xticks([0,1]); ax_cm.set_xticklabels(clases, rotation=15, fontsize=9)
ax_cm.set_yticks([0,1]); ax_cm.set_yticklabels(clases, fontsize=9)
for i in range(2):
    for j in range(2):
        ax_cm.text(j, i, str(cm[i,j]), ha='center', va='center',
                   fontsize=14, fontweight='bold',
                   color='white' if cm[i,j] > cm.max()/2 else 'black')
ax_cm.set_title(f'Matriz de confusión\n{MEJOR_MODELO} + {MEJOR_ESTRATEGIA} (test)',
                fontweight='bold')
ax_cm.set_ylabel('Real'); ax_cm.set_xlabel('Predicho')

plt.tight_layout()
graficos_b64['aprendizaje_confusion'] = figura_a_base64(fig)
plt.close(fig)
print('✅ Curva aprendizaje + matriz confusión generadas')

✅ Curva aprendizaje + matriz confusión generadas


In [11]:
# ============================================================================
# CELDA 10: SERIALIZACIÓN DE RESULTADOS
# ============================================================================

if not (RUTA_RESULTS / 'results_lineales_ext.parquet').exists():
    df_res.to_parquet(RUTA_RESULTS / 'results_lineales_ext.parquet', index=False)
    print('✅ results_lineales_ext.parquet guardado')
else:
    print('✅ results_lineales_ext.parquet ya existía')

df_res.to_excel(RUTA_RESULTS / 'results_lineales_ext.xlsx', index=False)

meta = {
    'fecha'            : datetime.now().isoformat(),
    'familia'          : FAMILIA,
    'modelos'          : MODELOS_EXT,
    'estrategias'      : ESTRATEGIAS,
    'n_combinaciones'  : len(claves_esperadas),
    'mejor_modelo'     : MEJOR_MODELO,
    'mejor_estrategia' : MEJOR_ESTRATEGIA,
    'mejor_auc_cv'     : round(float(df_cv.iloc[0]['auc_mean']), 4),
    'mejor_f1_cv'      : round(float(df_cv.iloc[0]['f1_mean']), 4),
    'nota_ridge'       : 'RidgeClassifier excluido — redundante con LogReg L2 (M01b). '
                         'SGDElasticNet lo generaliza con componente L1.',
}

with open(RUTA_RESULTS / 'results_lineales_ext.json', 'w', encoding='utf-8') as f:
    json.dump(meta, f, indent=2, ensure_ascii=False)

print(f'✅ {len(claves_esperadas)} pkl en {RUTA_MODELS}')
print(f'✅ results_lineales_ext.xlsx guardado')
print(f'✅ results_lineales_ext.json guardado')

✅ results_lineales_ext.parquet guardado
✅ 9 pkl en C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\05_modelado\models
✅ results_lineales_ext.xlsx guardado
✅ results_lineales_ext.json guardado


In [12]:
# ============================================================================
# CELDA 11: GENERAR HTML
# ============================================================================

RUTA_HTML_SALIDA = RUTA_HTML_F5 / 'm01c_lineales_ext.html'

kpis = [
    {'valor': '3',                     'titulo': 'Modelos'},
    {'valor': '3',                     'titulo': 'Estrategias'},
    {'valor': '9',                     'titulo': 'Combinaciones'},
    {'valor': str(N_TRIALS_OPTUNA),    'titulo': 'Trials Optuna'},
    {'valor': f'{df_cv.iloc[0]["auc_mean"]:.4f}', 'titulo': 'Mejor AUC CV'},
    {'valor': f'{df_cv.iloc[0]["f1_mean"]:.4f}',  'titulo': 'Mejor F1 CV'},
]
kpis_html = ''.join(
    f'<div class="kpi"><span class="kpi-valor">{k["valor"]}</span>'
    f'<span class="kpi-titulo">{k["titulo"]}</span></div>'
    for k in kpis
)

# Tabla resultados
filas_res = ''
for _, row in df_cv.iterrows():
    es_mejor = row['modelo'] == MEJOR_MODELO and row['estrategia'] == MEJOR_ESTRATEGIA
    bg = '#f0fff4' if es_mejor else ''
    bold = 'font-weight:bold' if es_mejor else ''
    filas_res += (
        f'<tr style="background:{bg};{bold}">'
        f'<td>{row["modelo"]}</td><td>{row["estrategia"]}</td>'
        f'<td>{row["auc_mean"]:.4f} ± {row["auc_std"]:.4f}</td>'
        f'<td>{row["f1_mean"]:.4f}</td>'
        f'<td>{row["precision_mean"]:.4f}</td>'
        f'<td>{row["recall_mean"]:.4f}</td></tr>'
    )

contenido = f'''
<section class="seccion">
  <h2>Modelos de esta familia</h2>
  <div class="kpis-grid">{kpis_html}</div>
  <table class="tabla-datos">
    <thead><tr><th>Modelo</th><th>Estrategia</th><th>AUC CV</th>
    <th>F1 CV</th><th>Precision</th><th>Recall</th></tr></thead>
    <tbody>{filas_res}</tbody>
  </table>
</section>

<section class="seccion">
  <h2>Justificación — por qué se excluye RidgeClassifier</h2>
  <p><code>RidgeClassifier</code> se omite porque su regularización L2 ya está
  representada por <code>LogisticRegression(penalty='l2')</code> en M01b,
  que implementa el mismo concepto matemático con diseño nativo para clasificación
  probabilística. <strong>SGDElasticNet lo generaliza</strong> añadiendo la
  componente L1 (selección automática de features) sin necesidad de calibración
  post-hoc. Incluir <code>RidgeClassifier</code> sería redundancia metodológica.</p>
</section>

<section class="seccion">
  <h2>Coeficientes — SGD vs SGDElasticNet</h2>
  <p>SGDElasticNet con <code>l1_ratio=0.5</code> combina L1 y L2.
  La componente L1 lleva algunos coeficientes exactamente a cero — selección
  automática de features. Ridge (L2 puro) nunca anula coeficientes.</p>
  <img src="data:image/png;base64,{graficos_b64['coeficientes']}" style="max-width:100%">
</section>

<section class="seccion">
  <h2>Curvas ROC y calibración</h2>
  <img src="data:image/png;base64,{graficos_b64['roc_calibracion']}" style="max-width:100%">
</section>

<section class="seccion">
  <h2>Curva de aprendizaje y matriz de confusión</h2>
  <img src="data:image/png;base64,{graficos_b64['aprendizaje_confusion']}" style="max-width:100%">
</section>
'''

render_pagina_desde_fichero('f5_m01c_lineales_ext.ipynb', contenido)
print(f'\n✅ HTML generado: {RUTA_HTML_SALIDA}')


✅ HTML generado: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase5\m01c_lineales_ext.html


In [13]:
# ============================================================================
# CELDA 12: RESUMEN FINAL
# ============================================================================

print('=' * 60)
print('RESUMEN FINAL — f5_m01c_lineales_ext')
print('=' * 60)
print(f'Familia     : Modelos lineales extendidos')
print(f'Modelos     : Perceptron · SGD · SGDElasticNet')
print(f'Estrategias : none · balanced · smote  (9 combinaciones)')
print(f'Excluido    : RidgeClassifier (redundante con LogReg L2 — ver HTML)')
print()
print(f'🏆 Mejor : {MEJOR_MODELO} + {MEJOR_ESTRATEGIA}')
print(f'   AUC CV = {df_cv.iloc[0]["auc_mean"]:.4f} ± {df_cv.iloc[0]["auc_std"]:.4f}')
print(f'   F1  CV = {df_cv.iloc[0]["f1_mean"]:.4f}')
print()
print('📁 Archivos:')
print(f'   data/05_modelado/results/results_lineales_ext.parquet')
print(f'   data/05_modelado/results/results_lineales_ext.xlsx')
print(f'   data/05_modelado/models/  (9 × .pkl)')
print(f'   docs/html/fase5/m01c_lineales_ext.html')
print()
print('➡️  Siguiente: f5_m01d_lineales.ipynb  (comparativa 7 modelos lineales)')

RESUMEN FINAL — f5_m01c_lineales_ext
Familia     : Modelos lineales extendidos
Modelos     : Perceptron · SGD · SGDElasticNet
Estrategias : none · balanced · smote  (9 combinaciones)
Excluido    : RidgeClassifier (redundante con LogReg L2 — ver HTML)

🏆 Mejor : SGDElasticNet + smote
   AUC CV = 0.9080 ± 0.0035
   F1  CV = 0.7499

📁 Archivos:
   data/05_modelado/results/results_lineales_ext.parquet
   data/05_modelado/results/results_lineales_ext.xlsx
   data/05_modelado/models/  (9 × .pkl)
   docs/html/fase5/m01c_lineales_ext.html

➡️  Siguiente: f5_m01d_lineales.ipynb  (comparativa 7 modelos lineales)
